In [ ]:
import serial
import csv
import time
import os

import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from scipy.interpolate import interp1d
import joblib

# Load scaler and model used during training
scaler = joblib.load("fitted_scaler1.pkl")
model = load_model("78.h5", compile=False)

# Sequence length used in the model
target_length = 50

def resample_sequence(sequence, target_len=50):
    seq_len = len(sequence)
    if seq_len == 0:
        return np.zeros((target_len, sequence.shape[1]))
    if seq_len == 1:
        return np.tile(sequence, (target_len, 1))
    
    new_time_index = np.linspace(0, seq_len - 1, target_len)
    interpolated = []
    for col_idx in range(sequence.shape[1]):
        f = interp1d(
            x=np.arange(seq_len),
            y=sequence[:, col_idx],
            kind='linear',
            fill_value="extrapolate"
        )
        interpolated.append(f(new_time_index))
    interpolated = np.array(interpolated).T
    return interpolated

def load_single_squat(filepath):
    df = pd.read_csv(filepath, header=0)
    df = df.dropna(how='any')
    if len(df) == 0:
        print("All rows are NaN")
        return None

    # Drop unnecessary columns
    df = df.drop(columns=["Timestamp", "IMU_ID", "Mag_X", "Mag_Y", "Mag_Z", "Heading", "Button1", "Button2"])

    # Check if sensor count is 6
    if df.shape[1] != 6:
        print("Sensor count is not 6; got:", df.shape[1])
        return None

    sensor_data = df.to_numpy()
    sensor_data_50 = resample_sequence(sensor_data, target_length)
    sensor_data_50 = scaler.transform(sensor_data_50)
    X = np.expand_dims(sensor_data_50, axis=0)  # (1, 50, 6)
    return X

def classify_squat(X):
    preds = model.predict(X)
    pred_label = np.argmax(preds, axis=1)[0] + 1
    confidence = np.max(preds)
    return pred_label, confidence

# Real-time Serial + Button detection -> CSV logging -> Immediate classification


SERIAL_PORT = "COM3"
BAUD_RATE = 115200
SAVE_FOLDER = "squat_logs"  # Folder to store squat CSV files

def main():
    # Create folder if it doesn't exist
    if not os.path.exists(SAVE_FOLDER):
        os.makedirs(SAVE_FOLDER)

    try:
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
        time.sleep(2)
    except serial.SerialException as e:
        print(f"Serial Error: {e}")
        return

    recording = False
    buffer_data = []

    print("Waiting for Calibration...")

    while True:
        try:
            line = ser.readline().decode("utf-8").strip()
            if not line:
                continue

            # 13 sensor values and buttons
            parts = line.split("\t")
            if len(parts) != 13:
                continue

            # Button1 and Button2 are the last two columns
            button1 = parts[-2]
            button2 = parts[-1]

            # If any button is pressed start recording
            if button1 == "1" or button2 == "1":
                if not recording:
                    recording = True
                    buffer_data = []
                    print("Button detected -> Start Recording")
                timestamp = time.strftime("%H:%M:%S")
                row = [timestamp] + parts
                buffer_data.append(row)
            else:
                if recording:
                    recording = False
                    print("Button released -> Squat done")
                    filename = time.strftime("squat_%Y%m%d_%H%M%S.csv")
                    filepath = os.path.join(SAVE_FOLDER, filename)

                    headers = [
                        "Timestamp", "IMU_ID", "Acc_X", "Acc_Y", "Acc_Z",
                        "Gyro_X", "Gyro_Y", "Gyro_Z",
                        "Mag_X", "Mag_Y", "Mag_Z", "Heading",
                        "Button1", "Button2"
                    ]
                    with open(filepath, mode="w", newline="") as f:
                        writer = csv.writer(f)
                        writer.writerow(headers)
                        writer.writerows(buffer_data)
                    print(f"    - {filepath} Saved ( {len(buffer_data)} Rows in Total)")

                    X = load_single_squat(filepath)
                    if X is not None:
                        label, conf = classify_squat(X)
                        print(f"    - Classifying result: label={label}, Confidence={conf:.2f}")
                        # Send the result to Arduino in the format "RESULT,label,confidence,sample_size"
                        result_str = f"RESULT,{label},{conf:.2f},{len(buffer_data)}\n"
                        ser.write(result_str.encode("utf-8"))
                    else:
                        print("    - CSV pre-processing failed; classification not possible")
                    buffer_data = []
        except KeyboardInterrupt:
            print("\nTerminated by user.")
            break

    ser.close()
    print("End program.")

if __name__ == "__main__":
    main()


Waiting for Calibration...
Button detected -> Start Recording
Button released -> Squat done
    - squat_logs\squat_20250321_122406.csv Saved ( 5 Rows in Total)
1/1 [==============================] - 1s 676ms/step
    - Classifying result: label=1, Confidence=0.79
Button detected -> Start Recording
Button released -> Squat done
    - squat_logs\squat_20250321_122419.csv Saved ( 40 Rows in Total)
1/1 [==============================] - 0s 28ms/step
    - Classifying result: label=1, Confidence=0.94
Button detected -> Start Recording
Button released -> Squat done
    - squat_logs\squat_20250321_122458.csv Saved ( 105 Rows in Total)
1/1 [==============================] - 0s 52ms/step
    - Classifying result: label=1, Confidence=0.94
Button detected -> Start Recording
Button released -> Squat done
    - squat_logs\squat_20250321_122458.csv Saved ( 105 Rows in Total)
1/1 [==============================] - 0s 45ms/step
    - Classifying result: label=1, Confidence=0.93
Button detected -> Star

SerialException: ClearCommError failed (PermissionError(13, 'El dispositivo no reconoce el comando.', None, 22))